In [1]:
import pathlib
import re
from functools import lru_cache

import numpy as np
import pandas as pd
import swifter

from rdkit import Chem

In [2]:
cwd = pathlib.Path().absolute()
QUANTUM_GREEN_DIR = pathlib.Path("/home/shared/projects/quantum_green")
PAPER_DIR = QUANTUM_GREEN_DIR / "paper" / "figure"
DATABASE_DIR = cwd.parent.parent / "data" / "kinetics"

In [3]:
pd.set_option("display.max_columns", None)


def head(df, n=2):
    display(df.head(n))
    print(f"Contains {len(df)} rows")


@lru_cache(maxsize=None)
def remapped_to_canonical(smiles):
    mol = Chem.MolFromSmiles(smiles)
    for atom in mol.GetAtoms():
        atom.SetAtomMapNum(0)
    return Chem.MolToSmiles(mol, isomericSmiles=True)


def remap_smiles(smiles):
    i = 0

    def _replace(_):
        nonlocal i
        i += 1
        return f":{i}"

    return re.sub(r":\d+", _replace, smiles)


def canonical_smiles(smiles):
    return remapped_to_canonical(remap_smiles(smiles))


def clean_rxn_smi(rxn_smi):
    return ">>".join(
        [
            ".".join([canonical_smiles(smi) for smi in category.split(".")])
            for category in rxn_smi.split(">>")
        ]
    )


def get_rxn_smi(row):
    return row["r1smi"] + "." + row["r2smi"] + ">>" + row["p1smi"] + "." + row["p2smi"]

In [4]:
rate_data = pd.read_csv(
    PAPER_DIR
    / "section_3_2_3_rate"
    / "quantum_green_ts_data_24september17_dft_opted_dlpno_sp_rates.csv",
)
rate_data["rxn_smi"] = rate_data.swifter.apply(get_rxn_smi, axis=1)
head(rate_data)

,Unnamed: 0,ts_dft_log_source_utf_8_sha512,multiplicity,r1smi,r2smi,p1smi,p2smi,std_xyz_str,r1_matched_std_xyz_str,r2_matched_std_xyz_str,p1_matched_std_xyz_str,p2_matched_std_xyz_str,ts_dft_frequencies,r1_dft_frequencies,r2_dft_frequencies,p1_dft_frequencies,p2_dft_frequencies,ts_dft_hartreefock_energy_hartree,r1_dft_hartreefock_energy_hartree,r2_dft_hartreefock_energy_hartree,p1_dft_hartreefock_energy_hartree,p2_dft_hartreefock_energy_hartree,ts_dlpno_sp_hartree,r1_dlpno_sp_hartree,r2_dlpno_sp_hartree,p1_dlpno_sp_hartree,p2_dlpno_sp_hartree,neg_freq,kinetics_low,k_298,p_reaction_low,m_reaction_low,p_rev_kinetics_low,m_rev_kinetics_low,kinetics_high,p_reaction_high,m_reaction_high,p_rev_kinetics_high,m_rev_kinetics_high,low_A,p_rev_low_A,m_rev_low_A,low_Ea,p_rev_low_Ea,m_rev_low_Ea,high_A,p_rev_high_A,m_rev_high_A,high_Ea,p_rev_high_Ea,m_rev_high_Ea,p_deltaHrxn298,m_deltaHrxn298,p_deltaGrxn298,m_deltaGrxn298,rxn_smi
0,0,4c648a435eb66f655a9afa7aebe1b6288cfbe5d4377ab7...,2,[O:1]([O:2])[H:3],[C:4]1([C:5]([C:8]([C:10](=[O:22])[H:23])([H:1...,[C:4]1([C:5]([C:8]([C:10](=[O:22])[H:23])([H:1...,[O:1]([O:2][H:11])[H:3],23\n\nO -0.296608 2.50034 -0.42834\nO 0.12862 ...,3\n\nO 0.055113 0.705073 0.0\nO 0.055113 -0.59...,20\n\nC 0.216362 1.782055 0.054566\nC -0.13150...,19\n\nC -0.008753 1.669215 -0.212931\nC 0.0510...,4\n\nO 0.0 0.708624 -0.054351\nO -0.0 -0.70862...,"[-1819.2074, 54.014, 69.9409, 78.7383, 95.0168...","[1268.6637, 1466.9774, 3673.0462]","[54.4468, 75.2434, 102.0769, 166.2486, 229.237...","[38.6623, 46.6427, 70.6254, 124.3627, 195.2907...","[355.51, 1029.1913, 1349.894, 1491.2134, 3851....",-499.513846,-150.736553,-348.800739,-348.140437,-151.376196,-499.420713,-150.773531,-348.675816,-348.013214,-151.4225,-1819.2074,"Arrhenius(A=(1.93009e+10,'cm^3/(mol*s)'), n=0,...",1.961906e-07,[O]O + CC(CC=O)C1CC1 <=> C[C](CC=O)C1CC1 + OO,[O]O + CC(CC=O)C1CC1 <=> C[C](CC=O)C1CC1 + OO,"Arrhenius(A=(23353.4,'m^3/(mol*s)'), n=-0.4985...","Arrhenius(A=(23353.4,'m^3/(mol*s)'), n=-0.4985...","Arrhenius(A=(2.46143e+12,'cm^3/(mol*s)'), n=0,...",[O]O + CC(CC=O)C1CC1 <=> C[C](CC=O)C1CC1 + OO,[O]O + CC(CC=O)C1CC1 <=> C[C](CC=O)C1CC1 + OO,"Arrhenius(A=(2.97825e+06,'m^3/(mol*s)'), n=-0....","Arrhenius(A=(2.97825e+06,'m^3/(mol*s)'), n=-0....",19300.901420,2.335344e+04,2.335344e+04,64517.736480,30906.955270,30906.955270,2.461429e+06,2.978245e+06,2.978245e+06,98788.712643,65177.931433,65177.931433,34171.85163,34171.85163,26963.412254,26963.412254,[O:1]([O:2])[H:3].[C:4]1([C:5]([C:8]([C:10](=[...
1,1,3fcdf61497a20d2e765655162f9a50bd8067a6ae90e90e...,2,[O:1]([O:2])[H:3],[C:4]1([O:11][C:5]([C:8]([O:12][C:10]([H:24])(...,[C:4]1([O:11][C:5]([C:8]([O:12][C:10]([H:25])[...,[O:1]([O:2][H:24])[H:3],26\n\nO 2.669129 -1.916241 0.102074\nO 3.62579...,3\n\nO 0.055113 0.705073 0.0\nO 0.055113 -0.59...,23\n\nC 2.355263 -1.592675 0.183484\nO 2.52464...,22\n\nC 1.79229 -1.9024 0.57597\nO 1.564071 -1...,4\n\nO 0.0 0.708624 -0.054351\nO -0.0 -0.70862...,"[-1780.9655, 22.7631, 39.1821, 48.2943, 53.310...","[1268.6637, 1466.9774, 3673.0462]","[41.7173, 46.7835, 79.0284, 116.919, 165.6573,...","[41.6898, 69.9592, 110.6483, 138.1279, 181.957...","[355.51, 1029.1913, 1349.894, 1491.2134, 3851....",-575.835651,-150.736553,-425.124273,-424.462568,-151.376196,-575.755980,-150.773531,-425.012518,-424.350164,-151.4225,-1780.9655,"Arrhenius(A=(4.46942e+10,'cm^3/(mol*s)'), n=0,...",1.102303e-07,[O]O + COCC(C)OC1CC1 <=> [CH2]OCC(C)OC1CC1 + OO,[O]O + COCC(C)OC1CC1 <=> [CH2]OCC(C)OC1CC1 + OO,"Arrhenius(A=(2.07496e+06,'m^3/(mol*s)'), n=-0....","Arrhenius(A=(2.07496e+06,'m^3/(mol*s)'), n=-0....","Arrhenius(A=(6.07844e+12,'cm^3/(mol*s)'), n=0,...",[O]O + COCC(C)OC1CC1 <=> [CH2]OCC(C)OC1CC1 + OO,[O]O + COCC(C)OC1CC1 <=> [CH2]OCC(C)OC1CC1 + OO,"Arrhenius(A=(2.82196e+08,'m^3/(mol*s)'), n=-0....","Arrhenius(A=(2.82196e+08,'m^3/(mol*s)'), n=-0....",44694.157891,2.074960e+06,2.074960e+06,68134.787555,35997.063187,35997.063187,6.078441e+06,2.821962e+08,2.821962e+08,102752.609475,70614.885108,7

Contains 166166 rows


In [5]:
thermo_data = pd.read_csv(
    PAPER_DIR
    / "section_3_2_1_thermo"
    / "quantum_green_species_data_24august14_dft_opted_dlpno_sp_thermos.csv"
)
for bac in ["p", "m"]:
    thermo_data[f"{bac}_G298"] = (
        thermo_data[f"{bac}_H298"] - 298.15 * thermo_data[f"{bac}_S298"]
    )
head(thermo_data)

,species_dft_log_source_utf_8_sha512,asmi,multiplicity,std_xyz_str,species_dft_frequencies,species_dft_hartreefock_energy_hartree,species_dlpno_sp_hartree,p_thermo,m_thermo,p_H298,m_H298,p_S298,m_S298,p_Cp300,m_Cp300,p_Cp400,m_Cp400,p_Cp500,m_Cp500,p_Cp600,m_Cp600,p_Cp800,m_Cp800,p_Cp1000,m_Cp1000,p_Cp1500,m_Cp1500,p_G298,m_G298
0,42d67c486eb53d22486a5d1da768e644bf644469198894...,[C:1]([C:2]1([H:13])[C:3]([H:14])([H:15])[N:4]...,2,17\n\nC 1.687648 -1.481461 0.769379\nC 1.14635...,"[69.0965, 133.1139, 185.0048, 247.5052, 269.23...",-434.321489,-434.224944,"Wilhoit(Cp0=(33.2579,'J/(mol*K)'), CpInf=(407....","Wilhoit(Cp0=(33.2579,'J/(mol*K)'), CpInf=(407....",89171.888145,92902.845218,384.472728,384.472728,138.524841,138.524841,175.864528,175.864528,208.654500,208.654500,236.432193,236.432193,278.945081,278.945081,308.548112,308.548112,350.979140,350.979140,-25458.655613,-21727.698541
1,7667c6f680d9065479d515666e4c3246f65704e4926140...,[C:1]([C:2]1([H:13])[C:3]([H:14])([H:15])[O:4]...,2,21\n\nC -2.800669 0.461226 0.774726\nC -1.7902...,"[40.2091, 100.6031, 169.247, 232.8355, 250.021...",-403.429122,-403.298238,"Wilhoit(Cp0=(33.2579,'J/(mol*K)'), CpInf=(507....","Wilhoit(Cp0=(33.2579,'J/(mol*K)'), CpInf=(507....",102076.181436,104632.984293,397.355792,397.355792,153.423259,153.423259,200.307331,200.307331,241.584659,241.584659,276.984827,276.984827,332.331738,332.331738,371.824840,371.824840,429.622128,429.622128,-16395.447972,-13838.645116


Contains 339428 rows


In [6]:
thermo_canonical_smiles = thermo_data["asmi"].swifter.apply(canonical_smiles)
mapping = {
    prop: dict(zip(thermo_canonical_smiles, thermo_data[prop]))
    for prop in ["m_H298", "m_G298", "p_H298", "p_G298"]
}
component_smiles = {
    component: rate_data[f"{component}smi"].swifter.apply(canonical_smiles)
    for component in ["r1", "r2", "p1", "p2"]
}
delta_rxn_298 = {
    prop: (
        component_smiles["p1"].apply(mapping[prop].get)
        + component_smiles["p2"].apply(mapping[prop].get)
        - component_smiles["r1"].apply(mapping[prop].get)
        - component_smiles["r2"].apply(mapping[prop].get)
    )
    for prop in ["m_H298", "m_G298", "p_H298", "p_G298"]
}

Pandas Apply:   0%|          | 0/339428 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/166166 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/166166 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/166166 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/166166 [00:00<?, ?it/s]

In [7]:
P2M = delta_rxn_298["m_H298"] - delta_rxn_298["p_H298"]
assert np.isclose(delta_rxn_298["m_G298"] - delta_rxn_298["p_G298"], P2M).all()

In [8]:
ts_key_char = pd.read_pickle(
    PAPER_DIR / "section_3_2_3_rate" / "ts_key_characteristics_july31a.pkl"
)
head(ts_key_char)

,rxn_smi,r1smi,r2smi,p1smi,p2smi,donor_element,donor_h_bond_distance,acceptor_element,acceptor_h_bond_distance,neg_freq,std_xyz_str,donor_atom_map_num,acceptor_atom_map_num,h_atom_map_num,ts_dft_log_source_utf_8_sha512,ts_angle_deg,ts_dist_donor_h,ts_dist_acceptor_h,r1_dft_zpe_scaled_hartree,r2_dft_zpe_scaled_hartree,p1_dft_zpe_scaled_hartree,p2_dft_zpe_scaled_hartree,ts_dft_zpe_scaled_hartree,r1_dlpno_sp_hartree,r2_dlpno_sp_hartree,p1_dlpno_sp_hartree,p2_dlpno_sp_hartree,ts_dlpno_sp_hartree,E0_r1,E0_r2,E0_p1,E0_p2,E0_ts,E0_r1_r2,E0_p1_p2,fwd_Hrxn_dlpno_sp_dft_zpe_scaled_hartree,rev_Hrxn_dlpno_sp_dft_zpe_scaled_hartree,fwd_barrier_dlpno_sp_dft_zpe_scaled_hartree,rev_barrier_dlpno_sp_dft_zpe_scaled_hartree,E0_r1_r2_kcal,E0_p1_p2_kcal,E0_ts_kcal,fwd_Hrxn_dlpno_sp_dft_zpe_scaled_kcal,rev_Hrxn_dlpno_sp_dft_zpe_scaled_kcal,fwd_barrier_dlpno_sp_dft_zpe_scaled_kcal,rev_barrier_dlpno_sp_dft_zpe_scaled_kcal,r1_canonical_smi
0,[O:1]([O:2])[H:3].[C:4]1([C:5]([C:8]([C:10](=[...,[O:1]([O:2])[H:3],[C:4]1([C:5]([C:8]([C:10](=[O:22])[H:23])([H:1...,[C:4]1([C:5]([C:8]([C:10](=[O:22])[H:23])([H:1...,[O:1]([O:2][H:11])[H:3],C,1.106773,O,0.965842,-1819.2074,23\n\nO -0.296608 2.50034 -0.42834\nO 0.12862 ...,5,2,11,4c648a435eb66f655a9afa7aebe1b6288cfbe5d4377ab7...,170.895816,1.312754,1.209684,0.014197,0.171556,0.157964,0.026431,0.182474,-150.773531,-348.675816,-348.013214,-151.4225,-499.420713,-150.759334,-348.50426,-347.85525,-151.396068,-499.238239,-499.263594,-499.251319,0.012275,-0.012275,0.025355,0.01308,-313292.635203,-313284.932511,-313276.724985,7.702692,-7.702692,15.910219,8.207526,[O]O
1,[O:1]([O:2])[H:3].[C:4]1([O:11][C:5]([C:8]([O:...,[O:1]([O:2])[H:3],[C:4]1([O:11][C:5]([C:8]([O:12][C:10]([H:24])(...,[C:4]1([O:11][C:5]([C:8]([O:12][C:10]([H:25])[...,[O:1]([O:2][H:24])[H:3],C,1.099011,O,0.965842,-1780.9655,26\n\nO 2.669129 -1.916241 0.102074\nO 3.62579...,10,2,24,3fcdf61497a20d2e765655162f9a50bd8067a6ae90e90e...,167.297745,1.332498,1.200348,0.014197,0.199164,0.185876,0.026431,0.210423,-150.773531,-425.012518,-424.350164,-151.4225,-575.755980,-150.759334,-424.813355,-424.164287,-151.396068,-575.545557,-575.572688,-575.560356,0.012333,-0.012333,0.027132,0.014799,-361177.31493,-361169.576075,-361160.289531,7.738855,-7.738855,17.0254,9.286544,[O]O


Contains 167237 rows


In [9]:
ts_clean_smiles = ts_key_char["rxn_smi"].swifter.apply(clean_rxn_smi)
ts_mappings = {
    "barrier": dict(
        zip(ts_clean_smiles, ts_key_char["fwd_barrier_dlpno_sp_dft_zpe_scaled_kcal"])
    ),
    "Hrxn": dict(
        zip(ts_clean_smiles, ts_key_char["fwd_Hrxn_dlpno_sp_dft_zpe_scaled_kcal"])
    ),
}

Pandas Apply:   0%|          | 0/167237 [00:00<?, ?it/s]

In [10]:
rate_clean_rxn_smiles = rate_data["rxn_smi"].swifter.apply(clean_rxn_smi)

kinetics_df = pd.DataFrame(
    {
        "rxn_smi": rate_clean_rxn_smiles,
        "k_298": rate_data["k_298"],
        "A_low": rate_data["low_A"],
        "Ea_low": rate_data["low_Ea"] / 4184,
        "A_high": rate_data["high_A"],
        "Ea_high": rate_data["high_Ea"] / 4184,
        "barrier": rate_clean_rxn_smiles.map(ts_mappings["barrier"]),
        "deltaHrxn": rate_clean_rxn_smiles.map(ts_mappings["Hrxn"]),
        "deltaHrxn298": delta_rxn_298["p_H298"] / 4184,
        "deltaGrxn298": delta_rxn_298["p_G298"] / 4184,
        "P2M": P2M / 4184,
    }
)
head(kinetics_df)

Pandas Apply:   0%|          | 0/166166 [00:00<?, ?it/s]

,rxn_smi,k_298,A_low,Ea_low,A_high,Ea_high,barrier,deltaHrxn,deltaHrxn298,deltaGrxn298,P2M
0,[O]O.CC(CC=O)C1CC1>>C[C](CC=O)C1CC1.OO,1.961906e-07,19300.901420,15.420109,2.461429e+06,23.611069,15.910219,7.702692,8.072324,6.348599,0.587593
1,[O]O.COCC(C)OC1CC1>>[CH2]OCC(C)OC1CC1.OO,1.102303e-07,44694.157891,16.284605,6.078441e+06,24.558463,17.025400,7.738855,7.786894,7.714869,0.652457


Contains 166166 rows


Looking for duplicates

In [11]:
kinetics_df[rate_clean_rxn_smiles.duplicated(keep=False)].sort_values(by="rxn_smi")

,rxn_smi,k_298,A_low,Ea_low,A_high,Ea_high,barrier,deltaHrxn,deltaHrxn298,deltaGrxn298,P2M
23746,[O]O.CCCc1nc(C)c[nH]1>>C[CH]Cc1nc(C)c[nH]1.OO,1.995910e-08,88431.113184,17.604535,9.092064e+06,25.521593,18.049820,11.564115,12.010234,10.572140,0.636948
50982,[O]O.CCCc1nc(C)c[nH]1>>C[CH]Cc1nc(C)c[nH]1.OO,1.171390e-08,83832.751812,17.901227,9.076999e+06,25.890575,18.049820,11.564115,12.010234,10.572140,0.636948
1736,[O]O.Cc1[nH]cnc1C=O>>Cc1[nH]cnc1[C]=O.OO,4.476911e-04,25467.490169,10.954568,2.641035e+06,18.847794,11.701590,4.886974,5.022356,4.285975,0.620976
28048,[O]O.Cc1[nH]cnc1C=O>>Cc1[nH]cnc1[C]=O.OO,6.236719e-04,26490.293900,10.772518,2.644290e+06,18.612342,11.701590,4.886974,5.022356,4.285975,0.620976
26312,[O]O.Cc1cc(C=O)n[nH]1>>Cc1cc([C]=O)n[nH]1.OO,5.964361e-05,27216.175834,12.247945,3.577528e+06,20.469567,13.886253,4.863133,4.983081,4.303859,0.619167
45920,[O]O.Cc1cc(C=O)n[nH]1>>Cc1cc([C]=O)n[nH]1.OO,6.217452e-05,195480.415751,13.392528,2.357858e+07,21.473775,13.886253,4.863133,4.983081,4.303859,0.619167
99824,[O]O.c1cc(C2CC2)n[nH]1>>C1=CC(C2CC2)=N[N]1.OO,9.367143e-12,635512.487498,23.370464,7.019250e+07,31.356097,23.482685,14.489843,14.865208,14.215183,0.360530
101950,[O]O.c1cc(C2CC2)n[nH]1>>C1=CC(C2CC2)=N[N]1.OO,7.771742e-12,628833.967265,23.478245,7.017883e+07,31.477803,23.482685,14.489843,14.865208,14.215183,0.360530


In [12]:
kinetics_df_dedup = kinetics_df.drop_duplicates(subset="rxn_smi", keep=False)

In [13]:
kinetics_df_dedup.to_csv(
    DATABASE_DIR / "quantumpioneer_kinetics_dataset.csv", index=False
)